In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from design_friendly.models import models_filepath
from py_wake import numpy as np
from py_wake.utils.gradients import autograd
from utils.misc import log_execution_time

# contains a single random layout from test set
test_graphs = torch.load(models_filepath + "sample.pt", weights_only=False)
ts_path = models_filepath + "best.ptnox.torchscript.pt"

/tmp/dgodi_4068900_hyd20251215/lib/python3.10/site-packages/tqdm_joblib/__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [3]:
from utils.get_flowmodel import get_flowmodel
from utils.iea22s import GEN22, IEA22s
from utils.to_graph import graph_maker_lut

wt = IEA22s()
wf_model = get_flowmodel(wt=wt)

grph = test_graphs[0]
X_fix = grph["pos"][:, 0]
Y_fix = grph["pos"][:, 1]
# del test_graphs
# wind rose
wds = np.arange(0, 360, 3)
wss = np.arange(3, 25, 2)
TI = 0.04  # single TI for now
tilt = 0
print(len(wds) * len(wss))
n_cpu = 1

lut_graphs = graph_maker_lut(
    x=X_fix,
    y=Y_fix,
    wds=wds,
    wss=wss,
    TI=0.04,  # single TI for now
    target_dicts=None,
    # per_turbines=False,
    connectivity="wake_aware",
    num_threads=32,
)

1320


Converting to graphs:   0%|          | 0/1320 [00:00<?, ?it/s]

  0%|          | 0/1320 [00:00<?, ?it/s]

INFO:utils.to_graph: generated 1320 graphs from 1320 cases


'generate_graphs | n=1 | 6.491 s | total=6 s'

'graph_maker_lut | n=1 | 6.493 s | total=6 s'

In [4]:
@log_execution_time
def power(yaw_ilk, X, Y, wds, wss, TI, n_cpu=1):
    tilt = 0
    _, _, power_ilk, _, _, _ = wf_model(
        x=X,
        y=Y,
        wd=wds,
        ws=wss,
        TI=TI,
        yaw=yaw_ilk,
        tilt=tilt,
        return_simulationResult=False,
        n_cpu=n_cpu,  # won't work with autograd() but maybe with MPI?
    )
    return power_ilk.sum()


dP_dyaw_xy = autograd(power, argnum=[0, 1, 2], vector_interdependence=True)


@log_execution_time
def partials_fn(g, gamma_np):
    pos = np.asarray(g["pos"], dtype=np.float32)  # (n_wt,2)
    X = pos[:, 0]
    Y = pos[:, 1]

    wd = float(g["meta"]["wd_deg"])
    ws = float(g["meta"]["ws"])
    TI_ = float(g["meta"]["ti"])

    wds_ = np.asarray([wd], dtype=np.float32)  # (1,)
    wss_ = np.asarray([ws], dtype=np.float32)  # (1,)
    yaw_ilk = np.asarray(gamma_np, dtype=np.float32)[:, None, None]  # (n_wt,1,1)

    dP_dyaw, dP_dX, dP_dY = dP_dyaw_xy(yaw_ilk, X, Y, wds_, wss_, TI_)

    dP_dgamma_vec = np.asarray(dP_dyaw[:, 0, 0], dtype=np.float32)  # (n_wt,)
    dP_dxy_mat = np.stack(
        [np.asarray(dP_dX, dtype=np.float32), np.asarray(dP_dY, dtype=np.float32)],
        axis=1,
    )  # (n_wt,2)

    return {"dP_dgamma": dP_dgamma_vec, "dP_dxy": dP_dxy_mat}

In [5]:
from design_friendly.utils.grads import (
    gradP_torchscript_vjp_xy_inflowgrid_prepared,
    make_dP_dz_inflowgrid,
    prepare_gradP_vjp_xy,
)

batch_size = len(lut_graphs)  # there seems to be a bug for batching (at assertion)
dP_dz = make_dP_dz_inflowgrid(wf_model)
prepared = prepare_gradP_vjp_xy(lut_graphs, batch_size=batch_size, edge_uv_cols=(0, 1))
dP_dxy, gamma = gradP_torchscript_vjp_xy_inflowgrid_prepared(
    ts_path,
    prepared,
    dP_dz,
    gamma_col=-1,
    uv_scale=1.0,
    return_gamma=True,
)

'make_dP_dz_inflowgrid | n=1 | 0.000 s | total=0 s'

'prepare_gradP_vjp_xy | n=1 | 0.310 s | total=0 s'

Converting to graphs:   0%|          | 0/1320 [00:11<?, ?it/s]


'partials_fn_inflowgrid_batch | n=1 | 43.556 s | total=44 s'

'gradP_torchscript_vjp_xy_inflowgrid_prepared | n=1 | 46.056 s | total=46 s'